In [1]:
import getpass
!pip install dbrepo==1.13.3

Defaulting to user installation because normal site-packages is not writeable
  Using cached dbrepo-1.13.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached pika-1.4.0-py3-none-any.whl.metadata (13 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.13-py3-none-any.whl.metadata (8.0 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached annotated_types-0.7

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import requests
import json
import pandas as pd
from getpass import getpass

In [10]:
from dbrepo.RestClient import RestClient
from getpass import getpass

username = "Bernh"
password = getpass()
auth = (username, password)

headers = {
    "Content-Type": "application/json"
}

BASE_URL = "https://test.dbrepo.tuwien.ac.at/api/v1"

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username=username,
    password=password
)

containers = client.get_containers()

for c in containers:
    print(c)

id='6cfb3b8e-1792-4e46-871a-f3d103527203' name='mariadb-galera:11.3.2' image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False) internal_name='mariadb_11_3_2' running=None hash=None


In [ ]:
CONTAINER_ID = "6cfb3b8e-1792-4e46-871a-f3d103527203"
database = client.create_database(
    "Vienna Demographic Forecasting",
    CONTAINER_ID,
    is_public=True,
    is_schema_public=True
)

In [15]:
databases = client.get_databases()

for db in databases:
    print(db)

id='38707917-e942-45c3-a3dd-d2bfc1c106af' name='Vienna Demographic Forecasting' contact=UserBrief(username='bernh', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) owned_by='bernh' internal_name='vienna_demographic_forecasting_2e4d' is_public=True is_schema_public=True identifiers=[] preview_image=None description=None
id='c77e4bfb-7f16-4a47-924b-430778476562' name='vienna_weather_wet_months' contact=UserBrief(username='azra1558', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) owned_by='azra1558' internal_name='vienna_weather_wet_months_cquf' is_public=False is_schema_public=True identifiers=[] preview_image=None description=None
id='81d82941-cae0-4cba-b27d-1bd883dd713a' name='data_stew_grp22_air_quality' contact=UserBrief(username='12450741', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) owned_by='12450741' internal_name='data_stew_grp22_air_quality_gvui' is_publi

In [13]:
database_id = "38707917-e942-45c3-a3dd-d2bfc1c106af"

In [ ]:
district_table = client.create_table(
    database_id=database_id,
    name="District",
    is_public=True,
    is_schema_public=True,
    dataframe=None,
    description="Administrative districts of Vienna based on Statistik Austria regional classification",
    with_data=False
)

In [18]:
district_payload = {
    "name": "District",
    "description": "Administrative districts of Vienna based on Statistik Austria regional classification",
    "columns": [
        {
            "name": "district_code",
            "type": "int",
            "size": None,
            "d": None,
            "description": "Unique identifier of Vienna district",
            "enums": None,
            "sets": None,
            "index_length": None,
            "null_allowed": False,
            "concept_uri": None,
            "unit_uri": None
        },
        {
            "name": "nuts_code",
            "type": "varchar",
            "size": 10,
            "d": None,
            "description": "NUTS regional classification code",
            "enums": None,
            "sets": None,
            "index_length": None,
            "null_allowed": False,
            "concept_uri": None,
            "unit_uri": None
        }
    ],
    "constraints": {
        "uniques": [],
        "checks": [],
        "foreign_keys": [],
        "primary_key": [
            "district_code"
        ]
    },
    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/table",
    auth=auth,
    headers=headers,
    json=district_payload
)

print(response.status_code)
print(response.text)

201
{"id":"682a018c-4a31-4324-8f52-40b13139f013","name":"District","description":"Administrative districts of Vienna based on Statistik Austria regional classification","database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"district","is_versioned":true,"is_public":true,"is_schema_public":true,"owned_by":"bernh"}


In [22]:
sex_payload = {
    "name": "Sex",
    "description": "Lookup table describing demographic sex categories",
    "columns": [
        {
            "name": "sex_id",
            "type": "int",
            "description": "Encoded sex identifier",
            "null_allowed": False
        },
        {
            "name": "label",
            "type": "varchar",
            "size": 20,
            "description": "Human-readable sex category label",
            "null_allowed": False
        }
    ],
    "constraints": {
        "uniques": [],
        "checks": [],
        "foreign_keys": [],
        "primary_key": [
            "sex_id"
        ]
    },
    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/table",
    auth=auth,
    headers=headers,
    json=sex_payload
)

print(response.status_code)
print(response.text)

201
{"id":"c039e55e-e5de-4603-b484-9b2c40f33658","name":"Sex","description":"Lookup table describing demographic sex categories","database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"sex","is_versioned":true,"is_public":true,"is_schema_public":true,"owned_by":"bernh"}


In [24]:
age_group_payload = {
    "name": "Age_group",
    "description": "Lookup table containing demographic age categories in 5-year intervals",
    "columns": [
        {
            "name": "age_group_id",
            "type": "int",
            "description": "Encoded age group identifier",
            "null_allowed": False
        },
        {
            "name": "age_range",
            "type": "varchar",
            "size": 20,
            "description": "Human-readable age interval",
            "null_allowed": False
        }
    ],
    "constraints": {
        "uniques": [],
        "checks": [],
        "foreign_keys": [],
        "primary_key": [
            "age_group_id"
        ]
    },
    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/table",
    auth=auth,
    headers=headers,
    json=age_group_payload
)

print(response.status_code)
print(response.text)

201
{"id":"a2333bf4-4ce2-42e6-a11d-32f9213b6488","name":"Age_group","description":"Lookup table containing demographic age categories in 5-year intervals","database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"age_group","is_versioned":true,"is_public":true,"is_schema_public":true,"owned_by":"bernh"}


In [25]:
nationality_payload = {
    "name": "Nationality_group",
    "description": "Lookup table containing grouped nationality categories used in demographic statistics",
    "columns": [
        {
            "name": "nationality_code",
            "type": "varchar",
            "size": 10,
            "description": "Nationality category code",
            "null_allowed": False
        },
        {
            "name": "description",
            "type": "varchar",
            "size": 100,
            "description": "Human-readable nationality category description",
            "null_allowed": True
        }
    ],
    "constraints": {
        "uniques": [],
        "checks": [],
        "foreign_keys": [],
        "primary_key": [
            "nationality_code"
        ]
    },
    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/table",
    auth=auth,
    headers=headers,
    json=nationality_payload
)

print(response.status_code)
print(response.text)

201
{"id":"19a15041-62dc-4676-992a-ea5acdb71ce3","name":"Nationality_group","description":"Lookup table containing grouped nationality categories used in demographic statistics","database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"nationality_group","is_versioned":true,"is_public":true,"is_schema_public":true,"owned_by":"bernh"}


In [26]:
time_payload = {
    "name": "Time_dimension",
    "description": "Temporal dimension table representing yearly reference dates",
    "columns": [
        {
            "name": "year",
            "type": "int",
            "description": "Reference year",
            "null_allowed": False
        },
        {
            "name": "reference_date",
            "type": "date",
            "description": "Reference date associated with the year",
            "null_allowed": False
        }
    ],
    "constraints": {
        "uniques": [],
        "checks": [],
        "foreign_keys": [],
        "primary_key": [
            "year"
        ]
    },
    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/table",
    auth=auth,
    headers=headers,
    json=time_payload
)

print(response.status_code)
print(response.text)

201
{"id":"b4a40404-3f22-4d6f-a534-8bae624aa576","name":"Time_dimension","description":"Temporal dimension table representing yearly reference dates","database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"time_dimension","is_versioned":true,"is_public":true,"is_schema_public":true,"owned_by":"bernh"}


In [37]:
population_payload = {
    "name": "Population_statistics",
    "columns": [
        {
            "name": "population_count",
            "type": "int",
            "description": "Observed population count",
            "null_allowed": False
        },
        {
            "name": "nationality_code",
            "type": "varchar",
            "size": 10,
            "description": "Nationality group identifier",
            "null_allowed": False
        },
        {
            "name": "district_code",
            "type": "int",
            "description": "District identifier",
            "null_allowed": False
        },
        {
            "name": "year",
            "type": "int",
            "description": "Reference year",
            "null_allowed": False
        },
        {
            "name": "sex_id",
            "type": "int",
            "description": "Sex category identifier",
            "null_allowed": False
        },
        {
            "name": "age_group_id",
            "type": "int",
            "description": "Age group identifier",
            "null_allowed": False
        }
    ],
    "constraints": {
        "uniques": [],
        "checks": [],
        "primary_key": [
            "district_code",
            "year",
            "sex_id",
            "age_group_id",
            "nationality_code"
        ],
        "foreign_keys": [
            {
                "columns": ["district_code"],
                "referenced_table": "district",
                "referenced_columns": ["district_code"]
            },
            {
                "columns": ["year"],
                "referenced_table": "time_dimension",
                "referenced_columns": ["year"]
            },
            {
                "columns": ["sex_id"],
                "referenced_table": "sex",
                "referenced_columns": ["sex_id"]
            },
            {
                "columns": ["age_group_id"],
                "referenced_table": "age_group",
                "referenced_columns": ["age_group_id"]
            },
            {
                "columns": ["nationality_code"],
                "referenced_table": "nationality_group",
                "referenced_columns": ["nationality_code"]
            }
]
    },
    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/table",
    auth=auth,
    headers=headers,
    json=population_payload
)

print(response.status_code)
print(response.text)

201
{"id":"8882a560-da44-4bb9-b14d-28ed91512c4b","name":"Population_statistics","description":null,"database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"population_statistics","is_versioned":true,"is_public":true,"is_schema_public":true,"owned_by":"bernh"}
